# Questions we can respond to

| Analysis                                                  | Question                                                          |
| --------------------------------------------------------- | ----------------------------------------------------------------- |
| `vote_average` ↔ `critic_score_percentage`                | Do TMDB users and RT critics rate movies in the same way?         |
| `vote_average` ↔ `audience_score_percentage`              | Do TMDB users and RT audiences rate movies in the same way?       |
| `critic_score_percentage` ↔ `audience_score_percentage`   | Do critics and audiences agree?                                   |
| `popularity` ↔ RT scores                                  | Are the most popular movies also the best rated?                  |
| `vote_count` ↔ `audience_rating_count`                    | Do both platforms attract the same level of audience engagement?  |
| `runtime` ↔ ratings                                       | Does the length of a movie affect its ratings?                    |
| ratings by `genre` / `year`                               | Do ratings vary by genre or over time?                            |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# works whether the notebook is run from its own folder or from the project root
ROOT = os.getcwd()
if not os.path.exists(os.path.join(ROOT, "data")):
    ROOT = os.path.abspath(os.path.join(ROOT, "..", ".."))

df = pd.read_csv(os.path.join(ROOT, "data/merged/manual_merge/final_merged_manual_rotten_tmdb.csv"))

In [ ]:
df.head(5)

## Initial description of the dataset

In [ ]:
print("shape:", df.shape)
print("duplicated rows:", df.duplicated().sum())
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

## Data preparation for the analysis

Before analysing, we make sure the variables are usable:
1. check that the expected columns exist
2. make the RT columns numeric
3. build `runtime` (in minutes)
4. put the TMDB score on the same 0-100 scale as RT
5. build the critics/audience gap
6. make sure we have a `year` column

In [ ]:
expected = ["title", "popularity", "vote_average", "vote_count",
            "critic_score_percentage", "audience_score_percentage",
            "critic_reviews_count", "audience_rating_count",
            "runtime_hour", "runtime_minutes", "genre"]

missing = [c for c in expected if c not in df.columns]
print("Missing columns:", missing)   # must be [] -- otherwise fix the name in the cells below
print()
print(df.columns.tolist())

In [ ]:
def to_num(s):
    """Convert text such as '85%' or '1,250' into a number (anything else -> NaN)."""
    if pd.api.types.is_numeric_dtype(s):
        return s
    return pd.to_numeric(s.astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce")

for c in ["critic_score_percentage", "audience_score_percentage",
          "critic_reviews_count", "audience_rating_count",
          "runtime_hour", "runtime_minutes"]:
    df[c] = to_num(df[c])

# runtime in minutes (if only one of hour/minutes is known, the other counts as 0)
runtime = df["runtime_hour"].fillna(0) * 60 + df["runtime_minutes"].fillna(0)
df["runtime"] = runtime.where(df["runtime_hour"].notna() | df["runtime_minutes"].notna())
df.loc[df["runtime"] <= 0, "runtime"] = np.nan

# vote_average = 0 with 0 votes means "nobody voted", not a real 0/10
df.loc[df["vote_count"].fillna(0) == 0, "vote_average"] = np.nan

# TMDB score on the same scale as RT (0-100)
df["tmdb_score_100"] = df["vote_average"] * 10

# gap > 0 : critics rate the movie higher than the audience
df["critic_audience_gap"] = df["critic_score_percentage"] - df["audience_score_percentage"]

# year
if "year" not in df.columns:
    if "release_year" in df.columns:
        df["year"] = df["release_year"]
    else:
        df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year

df[["title", "year", "tmdb_score_100", "critic_score_percentage",
    "audience_score_percentage", "critic_audience_gap", "runtime"]].head()

### distributions

In [ ]:
df[["popularity", "vote_average", "vote_count",
    "critic_score_percentage", "audience_score_percentage", "runtime"]].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

### the three scores on the same scale (0-100)

In [ ]:
plt.figure(figsize=(10, 5))
bins = np.linspace(0, 100, 21)
plt.hist(df["tmdb_score_100"].dropna(), bins=bins, alpha=0.5, label="TMDB (vote_average x 10)")
plt.hist(df["critic_score_percentage"].dropna(), bins=bins, alpha=0.5, label="RT critics")
plt.hist(df["audience_score_percentage"].dropna(), bins=bins, alpha=0.5, label="RT audience")
plt.xlabel("Score (0-100)")
plt.ylabel("Number of movies")
plt.title("Distribution of the three scores")
plt.legend()
plt.show()

**Observation :** *(à compléter : forme des distributions, quelle source note le plus généreusement ?)*

### vote_average vs critic_score_percentage

In [ ]:
df.plot.scatter(x="vote_average", y="critic_score_percentage",
                title="TMDB vote_average vs RT critic score")
plt.show()

### vote_average vs audience_score_percentage

In [ ]:
df.plot.scatter(x="vote_average", y="audience_score_percentage",
                title="TMDB vote_average vs RT audience score")
plt.show()

### critics vs audience (RT)

The red dashed line is the perfect agreement line (same score from critics and audience). Points above it: audience higher than critics. Points below: critics higher than audience.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(df["critic_score_percentage"], df["audience_score_percentage"], alpha=0.7)
plt.plot([0, 100], [0, 100], "r--", label="perfect agreement")
plt.xlabel("RT critic score (%)")
plt.ylabel("RT audience score (%)")
plt.title("Critics vs Audience")
plt.legend()
plt.show()

### popularity vs RT scores

`popularity` is very skewed (a few blockbusters), so it is displayed on a log scale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df["popularity"], df["critic_score_percentage"], alpha=0.7)
axes[0].set_xscale("log")
axes[0].set_xlabel("TMDB popularity (log scale)")
axes[0].set_ylabel("RT critic score (%)")

axes[1].scatter(df["popularity"], df["audience_score_percentage"], alpha=0.7)
axes[1].set_xscale("log")
axes[1].set_xlabel("TMDB popularity (log scale)")
axes[1].set_ylabel("RT audience score (%)")

plt.suptitle("Popularity vs RT scores")
plt.tight_layout()
plt.show()

### vote_count vs audience_rating_count

In [ ]:
sub = df[(df["vote_count"] > 0) & (df["audience_rating_count"] > 0)]
plt.figure(figsize=(8, 6))
plt.scatter(sub["vote_count"], sub["audience_rating_count"], alpha=0.7)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("TMDB vote count (log scale)")
plt.ylabel("RT audience rating count (log scale)")
plt.title("Audience engagement: TMDB vs Rotten Tomatoes")
plt.show()

**Observation :** *(à compléter après avoir regardé les 5 nuages de points : tendance visible ? points isolés ?)*

### Numeric columns

In [ ]:
numeric_cols = [
    "popularity",
    "vote_average",
    "vote_count",
    "critic_score_percentage",
    "audience_score_percentage",
    "critic_reviews_count",
    "audience_rating_count",
    "runtime",
    "year",
]

#### correlation matrix

In [ ]:
def plot_corr(corr, title):
    plt.figure(figsize=(10, 8))
    plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.columns)), corr.columns)
    for i in range(len(corr)):
        for j in range(len(corr)):
            plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_corr(df[numeric_cols].corr(), "Correlation Matrix (Pearson)")

#### Spearman correlation

Pearson is sensitive to extreme values (blockbusters). Spearman only uses the ranking of the movies, so it is more robust here.

In [ ]:
plot_corr(df[numeric_cols].corr(method="spearman"), "Correlation Matrix (Spearman)")

#### key pairs (with the number of movies used)

In [ ]:
pairs = [
    ("vote_average", "critic_score_percentage"),
    ("vote_average", "audience_score_percentage"),
    ("critic_score_percentage", "audience_score_percentage"),
    ("popularity", "critic_score_percentage"),
    ("popularity", "audience_score_percentage"),
    ("vote_count", "audience_rating_count"),
    ("runtime", "vote_average"),
    ("runtime", "critic_score_percentage"),
    ("runtime", "audience_score_percentage"),
]

rows = []
for a, b in pairs:
    sub = df[[a, b]].dropna()
    rows.append({
        "variables": f"{a} <-> {b}",
        "n": len(sub),
        "pearson": sub[a].corr(sub[b]),
        "spearman": sub[a].corr(sub[b], method="spearman"),
    })

key_pairs = pd.DataFrame(rows).round(3)
key_pairs

**Interprétation :** *(à compléter avec les valeurs du tableau ci-dessus, en citant r ET n. Repère : |r| < 0.3 faible, 0.3 à 0.7 modérée, > 0.7 forte. Une corrélation n'est pas une causalité.)*

| Variables | Pearson | Spearman | n | Interpretation |
| --------- | ------: | -------: | -: | -------------- |
|           |         |          |    |                |

### Do critics and audiences agree?

In [ ]:
gap = df["critic_audience_gap"].dropna()

print("mean gap (critics - audience):", round(gap.mean(), 1))
print("median gap:", round(gap.median(), 1))
print("movies with |gap| > 20 points:", (gap.abs() > 20).sum(), "/", len(gap))

plt.figure(figsize=(8, 5))
plt.hist(gap, bins=20, edgecolor="black")
plt.axvline(0, color="red", linestyle="--")
plt.xlabel("critic score - audience score (points)")
plt.ylabel("Number of movies")
plt.title("Critics/audience gap")
plt.show()

a positive gap means that critics like the movie more than the audience does.

In [ ]:
cols = ["title", "year", "critic_score_percentage", "audience_score_percentage", "critic_audience_gap"]

print("Critics much higher than the audience:")
display(df.nlargest(5, "critic_audience_gap")[cols])

print("Audience much higher than the critics:")
display(df.nsmallest(5, "critic_audience_gap")[cols])

**Observation :** *(à compléter : écart moyen, qui est plus sévère ? Pourquoi ces films divergent-ils (genre, type de film) ?)*

### outliers

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot([df["tmdb_score_100"].dropna(),
             df["critic_score_percentage"].dropna(),
             df["audience_score_percentage"].dropna()])
plt.xticks([1, 2, 3], ["TMDB (x10)", "RT critics", "RT audience"])
plt.ylabel("Score (0-100)")
plt.title("Scores - Outliers")
plt.show()

In [ ]:
df.nlargest(10, "popularity")[
    ["title", "year", "popularity", "vote_count",
     "tmdb_score_100", "critic_score_percentage", "audience_score_percentage"]
]

**Observation :** *(à compléter : ces films populaires sont-ils aussi bien notés ? Y a-t-il des films très populaires mais mal notés ?)*

### runtime vs ratings

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df["runtime"], df["tmdb_score_100"], alpha=0.7)
axes[0].set_ylabel("TMDB score (x10)")
axes[1].scatter(df["runtime"], df["critic_score_percentage"], alpha=0.7)
axes[1].set_ylabel("RT critic score (%)")
axes[2].scatter(df["runtime"], df["audience_score_percentage"], alpha=0.7)
axes[2].set_ylabel("RT audience score (%)")

for ax in axes:
    ax.set_xlabel("Runtime (minutes)")

plt.suptitle("Runtime vs ratings")
plt.tight_layout()
plt.show()

#### runtime groups

In [ ]:
bins = [0, 90, 105, 120, 150, 400]
labels = ["<90", "90-105", "105-120", "120-150", ">150"]

df["runtime_group"] = pd.cut(df["runtime"], bins=bins, labels=labels, include_lowest=True)

runtime_analysis = (
    df.groupby("runtime_group", observed=True)
      .agg(
          n_movies=("title", "count"),
          median_tmdb=("tmdb_score_100", "median"),
          median_critic=("critic_score_percentage", "median"),
          median_audience=("audience_score_percentage", "median"),
      )
      .reset_index()
)

runtime_analysis

In [ ]:
x = np.arange(len(runtime_analysis))
width = 0.25

plt.figure(figsize=(10, 6))
plt.bar(x - width, runtime_analysis["median_tmdb"], width, label="TMDB (x10)")
plt.bar(x, runtime_analysis["median_critic"], width, label="RT critics")
plt.bar(x + width, runtime_analysis["median_audience"], width, label="RT audience")
plt.xticks(x, runtime_analysis["runtime_group"].astype(str))
plt.xlabel("Runtime group (minutes)")
plt.ylabel("Median score (0-100)")
plt.title("Median scores by runtime group")
plt.legend()
plt.tight_layout()
plt.show()

**Observation :** *(à compléter : regarder aussi n_movies de chaque groupe, un groupe avec 3 films ne prouve rien)*

### ratings by year

In [ ]:
year_analysis = (
    df.groupby("year")
      .agg(
          n_movies=("title", "count"),
          median_tmdb=("tmdb_score_100", "median"),
          median_critic=("critic_score_percentage", "median"),
          median_audience=("audience_score_percentage", "median"),
          median_gap=("critic_audience_gap", "median"),
      )
      .sort_index()
)

year_analysis

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(year_analysis.index, year_analysis["median_tmdb"], marker="o", label="TMDB (x10)")
plt.plot(year_analysis.index, year_analysis["median_critic"], marker="o", label="RT critics")
plt.plot(year_analysis.index, year_analysis["median_audience"], marker="o", label="RT audience")
plt.xlabel("Year")
plt.ylabel("Median score (0-100)")
plt.title("Median Scores by Year")
plt.xticks(year_analysis.index, rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

**Observation :** *(à compléter : regarder n_movies par année avant de conclure)*

### ratings by genre

First, check how the genre is written in the file.

In [ ]:
df["genre"].head(10)

A movie has several genres, so we split them: one row per (movie, genre). A movie therefore counts once in each of its genres.

In [ ]:
genre_list = (df["genre"].astype(str)
                .str.replace(r"[\[\]'\"]", "", regex=True)
                .str.split(","))

df_genre = df.assign(genre_single=genre_list).explode("genre_single")
df_genre["genre_single"] = df_genre["genre_single"].str.strip()
df_genre = df_genre[~df_genre["genre_single"].isin(["", "nan", "None"])]

genre_analysis = (
    df_genre.groupby("genre_single")
            .agg(
                n_movies=("title", "count"),
                median_tmdb=("tmdb_score_100", "median"),
                median_critic=("critic_score_percentage", "median"),
                median_audience=("audience_score_percentage", "median"),
            )
            .query("n_movies >= 5")            # ignore genres with too few movies
            .sort_values("median_audience")
)

genre_analysis

In [ ]:
genre_analysis[["median_tmdb", "median_critic", "median_audience"]].plot.barh(figsize=(10, 8))
plt.xlabel("Median score (0-100)")
plt.ylabel("Genre")
plt.title("Median Scores by Genre (genres with at least 5 movies)")
plt.tight_layout()
plt.show()

**Observation :** *(à compléter : quels genres sont les mieux/moins bien notés ? Critiques et public classent-ils les genres de la même façon ?)*

## Summary of findings

1. *(TMDB vs critics RT : corrélation, force, n)*

2. *(TMDB vs audience RT : corrélation, force, n)*

3. *(Critics vs audience : écart moyen, films qui divergent)*

4. *(Popularity vs scores)*

5. *(Runtime / genre / year)*

## Limitations

1. Only the movies successfully matched between TMDB and Rotten Tomatoes are in this dataset. It is a subset built by our matching process (title similarity + year), probably biased towards well-known movies.
2. The sample is small (see `df.shape` and the `n` column of the key pairs table): correlation coefficients are unstable and a few movies can change them.
3. The scales are not the same: TMDB is an average on 0-10, RT scores are percentages of positive reviews/ratings. We compare trends (Spearman), not exact values.
4. Missing values in some RT columns (runtime, counts) reduce `n` for the analyses that use them.
5. `popularity` is a snapshot at the date of the scraping and favours recent movies. `audience_rating_count` may be approximate (see the `audience_rating_count_operator` column): check it before interpreting.
6. Correlation is not causation.